In [6]:
import os
from pathlib import Path
import pandas as pd
from ollama import Client
from dotenv import load_dotenv

In [7]:
data_path = Path("../../data/processed/herd_mentality_events_processed_v1.csv").resolve()
processed_path = Path("../../data/processed/herd_mentality_events_processed_v2.csv").resolve()

load_dotenv()
OLLAMA_API_KEY = os.getenv("OLLAMA_API_KEY")

In [8]:
df = pd.read_csv(data_path)
df

,Year,Event Name,Continent,Event Description
0,1925,South African Rand mine strike,Africa,"In 1925, thousands of African gold‑mine worker..."
1,1926,Birth of Pan-African Congress,Africa,"In 1926, African intellectuals and diaspora le..."
2,1927,Foundation of ANC Youth League (SA),Africa,"In 1927, young activists in South Africa found..."
3,1928,South African Black political conference,Africa,"In 1928, black political leaders from across S..."
4,1930,Mass migration for economic reasons in Sahel,Africa,In 1930 a severe drought combined with the glo...
...,...,...,...,...
620,2021,Papua volcano eruption response,Australia/Oceania,"In December 2021, Mount Manam in Papua New Gui..."
621,2022,Fiji cyclone mass evacuations,Australia/Oceania,"In February 2022, Fiji’s Meteorological Servic..."
622,2023,Indigenous Voice to Parliament activism (Austr...,Australia/Oceania,"Throughout 2023, Australia saw a nationwide su..."
623,2024,Pacific Islands anti-mining protests,Australia/Oceania,"In 2024, island communities across the Pacific..."


In [9]:
OLLAMA_API_BASE = "https://ollama.com"
OLLAMA_MODEL = "gpt-oss:120b"

# Initialize Ollama Cloud client with authentication
# NOTE: Verify your API key at https://ollama.com/settings/keys
# The API key should be a simple string, not an SSH key format
client = Client(host=OLLAMA_API_BASE, headers={'Authorization': 'Bearer ' + OLLAMA_API_KEY})

def run_ollama_chat(prompt: str,
                    model: str = OLLAMA_MODEL,
                    temperature: float = 0.1,
                    max_tokens: int | None = 1000) -> str:
    """Send a chat prompt to Ollama Cloud and return the response text."""
    response = client.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={
            "temperature": temperature,
            "num_predict": max_tokens if max_tokens else None
        }
    )
    # Ollama Client returns ChatResponse with message.content, not choices[0].message.content
    return response['message']['content']

result = run_ollama_chat("What is the capital of France?. Answer in 2 to 3 words.")
print(result)

Paris, France


In [10]:
def formating_prompt(year, event_name, continent, description):
    prompt_country = f"""
    You are an expert geopolitical historian AI tasked with identifying the most likely country associated with a historical event.

    <<INPUTS>>
    Year: {year}
    Event Name: {event_name}
    Continent: {continent}
    Event Description: {description}

    <<OBJECTIVE>>
    Determine the single country most central to the specified event.

    <<THINKING STRATEGY>>
    1. Parse the event details to recall historical context and principal actors.
    2. Validate that the inferred country aligns with the provided continent and description.
    3. If multiple countries are involved, choose the one most directly responsible for or affected by the event.
    4. If information is insufficient or conflicting, respond with "Unknown".
    Reason internally through these steps before answering, but do not reveal your chain of thought.

    <<RESPONSE CONSTRAINTS>>
    - Output must follow the format `Country: <Country Name>` using 2 to 3 words.
    - If uncertain, respond exactly with `Country: Unknown`.
    - Do not include any additional commentary, justification, or formatting.

    <<FEW-SHOT EXAMPLES>>
    Example 1
    Year: 1927
    Event Name: Foundation of ANC Youth League (SA)
    Continent: Africa
    Event Description: The ANC Youth League formed to energize anti-colonial activism, mobilizing young South Africans against segregationist policies.
    Final Answer: Country: South Africa

    Example 2
    Year: 1935
    Event Name: Italian invasion of Ethiopia (mobilizations)
    Continent: Africa
    Event Description: Mussolini's forces advanced into Ethiopia, facing fierce resistance and prompting global condemnation of Italian aggression.
    Final Answer: Country: Ethiopia

    Example 3
    Year: 1927
    Event Name: Papua land protests
    Continent: Australia/Oceania
    Event Description: Indigenous Papuans protested colonial land seizures, demanding recognition of traditional ownership and rights under Australian administration.
    Final Answer: Country: Papua New Guinea

    Example 4
    Year: 1969
    Event Name: India-Pakistan war (mass mobilization)
    Continent: Asia
    Event Description: Military forces massed along the border as India intervened in East Pakistan, shaping the conflict that birthed Bangladesh.
    Final Answer: Country: India

    Example 5
    Year: 1945
    Event Name: End of WWII: mass repatriations
    Continent: Asia
    Event Description: Allied authorities orchestrated the return of millions across Asia as Japan surrendered and occupation zones reorganized.
    Final Answer: Country: Japan

    Example 6
    Year: 1926
    Event Name: Birth of Pan-African Congress
    Continent: Africa
    Event Description: The Birth of Pan-African Congress marked the formal establishment of the first Pan-African Congress in 1926, uniting African leaders to discuss unity and liberation across the continent. It laid the groundwork for future pan-African movements.
    Final Answer: Country: Pan-Africa

    Provide only the final answer in the specified format:
    Country: <Country Name>
    """
    return prompt_country

In [ ]:
def parse_country_from_response(response_text: str) -> str:
    """Extract the country value from a `Country: <value>` response."""
    cleaned = response_text.strip().strip("`")

    for line in cleaned.splitlines():
        stripped = line.strip()
        if not stripped:
            continue
        if stripped.lower().startswith("country:"):
            country = stripped.split(":", 1)[1].strip().strip('"')
            return country if country else "Unknown"

    return "Unknown"


def enrich_with_countries(dataframe: pd.DataFrame, start_idx: int = 0, end_idx: int | None = None) -> pd.DataFrame:
    """Enrich dataframe slice with predicted countries (supports partial ranges)."""
    if end_idx is None:
        end_idx = len(dataframe)

    subset_df = dataframe.iloc[start_idx:end_idx].copy()
    countries: list[str] = []

    for i, row in subset_df.iterrows():
        year = row["Year"]
        event_name = row["Event Name"]
        continent = row["Continent"]
        description = row.get("Event Description", "")
        if pd.isna(description):
            description = ""
        prompt = formating_prompt(year, event_name, continent, description)
        try:
            response_text = run_ollama_chat(prompt)
            print(response_text)
            country = parse_country_from_response(response_text)
            countries.append(country)
            print(f"record {start_idx + i + 1} of {len(dataframe)} processed: {country}")
        except Exception as e:
            print(f"ERROR processing record {start_idx + i + 1}: {e}")
            descriptions.append("ERROR: Failed to generate description")

    subset_df["Country"] = countries
    return subset_df


def process_with_batch_saving(df: pd.DataFrame, batch_size: int = 100):
    """
    Process data in batches, save each batch, then merge automatically.
    Automatically resumes from where it left off if interrupted.
    Returns the final merged dataframe (or None if not all batches finished).
    """
    import time
    from pathlib import Path

    batch_dir = processed_path.parent / "batches_country"
    batch_dir.mkdir(exist_ok=True)

    total = len(df)
    num_batches = (total + batch_size - 1) // batch_size

    print(f"Processing {total} records in {num_batches} batches of {batch_size}...\n")

    all_batches: list[Path] = []
    skipped_count = 0

    for batch_num in range(num_batches):
        start = batch_num * batch_size
        end = min(start + batch_size, total)
        batch_file = batch_dir / f"country_batch_{batch_num + 1:03d}.csv"

        # Resume: skip batches already completed
        if batch_file.exists():
            print(f"⏭ Batch {batch_num + 1}/{num_batches} already exists, skipping...")
            all_batches.append(batch_file)
            skipped_count += 1
            continue

        print(f"Batch {batch_num + 1}/{num_batches} (records {start + 1}-{end})...")
        
        try:
            batch_df = enrich_with_countries(df, start_idx=start, end_idx=end)
            batch_df.to_csv(batch_file, index=False)
            all_batches.append(batch_file)
            print(f"✓ Saved: {batch_file.name}\n")
        except Exception as e:
            print(f"✗ ERROR in batch {batch_num + 1}: {e}")
            print(f"  You can resume processing later - batches 1-{batch_num} are saved.\n")
            raise

        if batch_num < num_batches - 1:
            time.sleep(2)  # small pause to respect rate limits

    if skipped_count > 0:
        print(f"\nResumed: Skipped {skipped_count} already-completed batches\n")

    # Only merge when all batches are complete
    if len(all_batches) == num_batches:
        print("Merging all batches...")
        merged = pd.concat([pd.read_csv(f) for f in all_batches], ignore_index=True)
        merged.to_csv(processed_path, index=False)
        print(f"✓ Done! Final file: {processed_path}")
        return merged

    print(f"\n⚠ Not all batches complete yet ({len(all_batches)}/{num_batches}). Re-run this cell to continue.")
    return None

In [ ]:
# Process data in batches (saves each batch and merges automatically)
# If processing stops, just re-run this cell - it will resume from where it left off.
df_enriched = process_with_batch_saving(df, batch_size=100)